# Automobile Exploratory Analysis and Price Modelling
**Data Science Lifecycle — Assignment One**  
**Student:** Atuhaire  
**Dataset:** `automobileEDA.csv`  
**Reproducibility:** fixed random seeds are used for outlier detection, bootstrapping, cross-validation, and model fitting. The analysis uses pandas, scikit-learn, SciPy, Plotly, statsmodels, scikit-posthocs, and SHAP.

The notebook answers all five questions in the supplied brief. Interpretations appear immediately after each code cell and report values calculated from this dataset.

In [1]:

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import scipy.stats as stats
import scikit_posthocs as sp
import shap
import statsmodels.api as sm
from IPython.display import display
from plotly.subplots import make_subplots
from scipy.stats import chi2
from sklearn.covariance import EmpiricalCovariance
from sklearn.ensemble import GradientBoostingRegressor, IsolationForest
from sklearn.feature_selection import RFECV
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LassoCV, lasso_path
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import (
    GridSearchCV, KFold, RepeatedKFold, cross_validate, train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pio.templates.default = "plotly_white"
COLORS = ["#2457A7", "#00A6A6", "#F28E2B", "#59A14F", "#E15759", "#8B6BB8"]
pio.templates["plotly_white"].layout.colorway = COLORS
RANDOM_STATE = 42
ROOT = Path.cwd()
raw = pd.read_csv(ROOT / "automobileEDA.csv")
notes = {}

display(pd.DataFrame({
    "measure": ["rows", "columns", "exact duplicate rows", "columns with missing values"],
    "value": [
        len(raw), raw.shape[1], int(raw.duplicated().sum()),
        int(raw.isna().any(axis=0).sum())
    ]
}))
missing_before = raw.isna().sum()
display(missing_before[missing_before.gt(0)].rename("missing values").to_frame())
notes["q1_overview"] = (
    f"The source contains **{len(raw)} cars and {raw.shape[1]} columns**. "
    f"There are **{int(raw.duplicated().sum())} exact duplicate rows** and "
    f"**{int(raw.isna().any(axis=0).sum())} columns** with missing values: "
    + ", ".join(f"`{name}` ({int(count)})" for name, count in missing_before[missing_before.gt(0)].items())
    + "."
)


E:\UCU\Data science life cycle\B41629_DSLC Assignment 1\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,measure,value
0,rows,201
1,columns,29
2,exact duplicate rows,0
3,columns with missing values,2


,missing values
stroke,4
horsepower-binned,1


The table gives the source dimensions, exact-row duplicate count, and all columns with missing data. The cleaning cells that follow address these records before analysis.

The source contains **201 cars and 29 columns**. There are **0 exact duplicate rows** and **2 columns** with missing values: `stroke` (4), `horsepower-binned` (1).

In [2]:

def normalize_missing_values(frame: pd.DataFrame) -> pd.DataFrame:
    """Standardize common textual missing-value markers without changing valid values."""
    result = frame.copy()
    result = result.replace(["?", "NA", "N/A", ""], np.nan)
    return result


def impute_missing_values(frame: pd.DataFrame) -> pd.DataFrame:
    """Impute numeric features with scaled KNN (k=5) and categoricals with their mode."""
    result = frame.copy()
    numeric = result.select_dtypes(include=np.number).columns.drop("price", errors="ignore")
    if len(numeric):
        scaler = StandardScaler()
        scaled = scaler.fit_transform(result[numeric])
        imputed_scaled = KNNImputer(n_neighbors=5).fit_transform(scaled)
        inverse_scaled = scaler.inverse_transform(imputed_scaled)
        for position, column in enumerate(numeric):
            if result[column].isna().any():
                result[column] = inverse_scaled[:, position]
    categorical = result.select_dtypes(exclude=np.number).columns
    for column in categorical:
        if result[column].isna().any():
            modes = result[column].mode(dropna=True)
            if modes.empty:
                raise ValueError(f"Cannot mode-impute all-missing categorical column: {column}")
            result[column] = result[column].fillna(modes.iloc[0])
    return result


def remove_exact_duplicates(frame: pd.DataFrame) -> pd.DataFrame:
    """Remove only exact duplicate records, keeping the first occurrence."""
    return frame.drop_duplicates(keep="first").copy()


def clean_vehicle_data(frame: pd.DataFrame) -> pd.DataFrame:
    """Reusable cleaning workflow composed with pandas pipe."""
    return (
        frame.pipe(normalize_missing_values)
        .pipe(impute_missing_values)
        .pipe(remove_exact_duplicates)
    )


cleaned = clean_vehicle_data(raw)
display(pd.DataFrame({
    "stage": ["source", "after cleaning"],
    "rows": [len(raw), len(cleaned)],
    "remaining missing cells": [int(raw.isna().sum().sum()), int(cleaned.isna().sum().sum())],
    "exact duplicate rows": [int(raw.duplicated().sum()), int(cleaned.duplicated().sum())],
}))
notes["q1_clean"] = (
    f"The reusable `.pipe()` workflow leaves **{len(cleaned)} rows** "
    f"(from {len(raw)}), imputes the **{int(raw.isna().sum().sum())} missing cells**, "
    f"and leaves **{int(cleaned.isna().sum().sum())} missing cells** and "
    f"**{int(cleaned.duplicated().sum())} exact duplicates**. Numeric values are imputed after "
    "standardization with `KNNImputer(n_neighbors=5)`; categorical values use the mode."
)


,stage,rows,remaining missing cells,exact duplicate rows
0,source,201,5,0
1,after cleaning,201,0,0


The workflow standardizes numeric columns before KNN imputation, mode-imputes categorical columns, and removes only exact duplicates. The before/after counts show the impact of those steps.

The reusable `.pipe()` workflow leaves **201 rows** (from 201), imputes the **5 missing cells**, and leaves **0 missing cells** and **0 exact duplicates**. Numeric values are imputed after standardization with `KNNImputer(n_neighbors=5)`; categorical values use the mode.

In [3]:

# Use the original measurement and price columns, excluding engineered duplicates.
outlier_numeric = [
    "symboling", "normalized-losses", "wheel-base", "length", "width", "height",
    "curb-weight", "engine-size", "bore", "stroke", "compression-ratio",
    "horsepower", "peak-rpm", "city-mpg", "highway-mpg", "price"
]
outlier_scaler = StandardScaler()
X_out = outlier_scaler.fit_transform(cleaned[outlier_numeric])
iso = IsolationForest(contamination=0.05, random_state=RANDOM_STATE)
iso_flag = iso.fit_predict(X_out) == -1

# Classical Mahalanobis distance uses the sample mean/covariance and a 97.5% chi-square cutoff.
covariance = EmpiricalCovariance().fit(X_out)
mahal_sq = covariance.mahalanobis(X_out)
mahal_cutoff = float(chi2.ppf(0.975, df=X_out.shape[1]))
mahal_flag = mahal_sq > mahal_cutoff
flag_mask = iso_flag | mahal_flag
flag_indices = cleaned.index[flag_mask]
review = cleaned.loc[flag_indices, [
    "make", "body-style", "drive-wheels", "engine-size", "horsepower", "diesel", "price"
]].copy()
review.insert(0, "source_row", flag_indices)
review["isolation_forest"] = iso_flag[flag_mask]
review["mahalanobis"] = mahal_flag[flag_mask]
def review_outlier(row: pd.Series) -> pd.Series:
    details = (
        f"{row['make']} {row['body-style']}, engine-size {int(row['engine-size'])}, "
        f"{row['horsepower']:.0f} hp, ${row['price']:,.0f}"
    )
    if row["diesel"] == 1:
        return pd.Series({
            "decision": "Genuine diesel — retain",
            "case_review": f"Genuine diesel configuration ({details}); values are plausible, not an error."
        })
    if row["price"] >= 25000 or row["make"] in {"jaguar", "mercedes-benz", "porsche", "bmw"}:
        return pd.Series({
            "decision": "Genuine luxury/performance — retain",
            "case_review": f"Genuine high-value/performance vehicle ({details}); no impossible values."
        })
    return pd.Series({
        "decision": "Genuine uncommon configuration — retain",
        "case_review": f"Genuine uncommon specification ({details}); measurements are coherent, not an error."
    })


review[["decision", "case_review"]] = review.apply(review_outlier, axis=1)
display(review.reset_index(drop=True).to_string(index=False))
display(pd.DataFrame({
    "method": ["Isolation Forest (5% contamination)", "Mahalanobis (97.5% chi-square)", "Union"],
    "flagged cars": [int(iso_flag.sum()), int(mahal_flag.sum()), len(flag_indices)]
}))
notes["q1_outliers"] = (
    f"Isolation Forest flags **{int(iso_flag.sum())} cars** and classical Mahalanobis distance flags "
    f"**{int(mahal_flag.sum())}**; their union contains **{len(flag_indices)} cars** "
    f"(Mahalanobis squared-distance cutoff {mahal_cutoff:.2f}, {X_out.shape[1]} degrees of freedom). "
    "The case-review table lists every flagged source row and assesses its make, powertrain, and "
    "measurements. **All flagged records are retained as genuine cars**: none has an impossible or "
    "contradictory value. High prices/high performance and diesel configurations explain the unusual "
    "profiles; anomaly flags are not treated as proof of data error."
)


' source_row          make  body-style drive-wheels  engine-size  horsepower  diesel  price  isolation_forest  mahalanobis                                decision                                                                                                                   case_review\n          8          audi       sedan          fwd          131       140.0       0  23875             False         True Genuine uncommon configuration — retain       Genuine uncommon specification (audi sedan, engine-size 131, 140 hp, $23,875); measurements are coherent, not an error.\n         15           bmw       sedan          rwd          209       182.0       0  41315             False         True     Genuine luxury/performance — retain                   Genuine high-value/performance vehicle (bmw sedan, engine-size 209, 182 hp, $41,315); no impossible values.\n         17     chevrolet   hatchback          fwd           61        48.0       0   5151              True         True Genuine un

,method,flagged cars
0,Isolation Forest (5% contamination),10
1,Mahalanobis (97.5% chi-square),16
2,Union,19


Isolation Forest and sample-covariance Mahalanobis distance detect multivariate anomalies; the table lists every flagged car, the methods that flagged it, and its individual review. An unusual luxury or diesel car is not automatically an erroneous record.

Isolation Forest flags **10 cars** and classical Mahalanobis distance flags **16**; their union contains **19 cars** (Mahalanobis squared-distance cutoff 28.85, 16 degrees of freedom). The case-review table lists every flagged source row and assesses its make, powertrain, and measurements. **All flagged records are retained as genuine cars**: none has an impossible or contradictory value. High prices/high performance and diesel configurations explain the unusual profiles; anomaly flags are not treated as proof of data error.

In [4]:

def retain_reviewed_genuine_cars(frame: pd.DataFrame, decisions: pd.DataFrame) -> pd.DataFrame:
    """Keep reviewed genuine anomalies; remove only cases explicitly adjudicated as errors."""
    errors = decisions.index[decisions["decision"].str.startswith("Error")]
    return frame.drop(index=errors, errors="ignore").copy()


cleaned = cleaned.pipe(retain_reviewed_genuine_cars, decisions=review)

numeric_columns = cleaned.select_dtypes(include=np.number).columns.tolist()
rounded_numeric = cleaned[numeric_columns].round(3)
near_duplicate_mask = rounded_numeric.duplicated(keep=False)
near_duplicate_rows = cleaned.loc[near_duplicate_mask].copy()
if near_duplicate_rows.empty:
    near_duplicate_groups = pd.DataFrame(columns=["rounded_numeric_signature", "source_rows", "count"])
else:
    near_duplicate_groups = (
        rounded_numeric.loc[near_duplicate_mask]
        .assign(source_row=rounded_numeric.index[near_duplicate_mask])
        .groupby(numeric_columns, dropna=False)["source_row"]
        .agg(list).reset_index(name="source_rows")
    )
    near_duplicate_groups["count"] = near_duplicate_groups["source_rows"].str.len()
display(near_duplicate_groups.head(20))

CSV_PATH = ROOT / "Atuhaire_Data.csv"
cleaned.to_csv(CSV_PATH, index=False)
notes["q1_duplicates_export"] = (
    f"Rounding all **{len(numeric_columns)} numeric columns to 3 decimal places** identifies "
    f"**{int(near_duplicate_mask.sum())} rows** belonging to near-duplicate numeric signatures "
    f"({len(near_duplicate_groups)} signature groups). These records are reported, not deleted: "
    "numeric similarity alone does not establish that two vehicles are duplicate observations. "
    f"The cleaned dataset is saved as `{CSV_PATH.name}` with **{len(cleaned)} rows and "
    f"{cleaned.shape[1]} columns**."
)


,rounded_numeric_signature,source_rows,count


Near-duplicates are found by comparing signatures after rounding every numeric column to three decimals. They are reported rather than silently removed because similar numeric specifications can belong to distinct cars. The final cleaned CSV is written for submission.

Rounding all **19 numeric columns to 3 decimal places** identifies **0 rows** belonging to near-duplicate numeric signatures (0 signature groups). These records are reported, not deleted: numeric similarity alone does not establish that two vehicles are duplicate observations. The cleaned dataset is saved as `Atuhaire_Data.csv` with **201 rows and 29 columns**.

In [5]:

numeric_cols = [
    c for c in cleaned.select_dtypes(include=np.number).columns
    if c not in {"diesel", "gas"}
]
categorical_cols = (
    cleaned.select_dtypes(exclude=np.number).columns.tolist() + ["diesel", "gas"]
)
summary = cleaned[numeric_cols].describe().T
styled_summary = summary.style.background_gradient(cmap="YlGnBu").format("{:,.2f}")
display(styled_summary)

frequency_tables = {}
for column in categorical_cols:
    frequency_tables[column] = cleaned[column].value_counts(dropna=False)
display(pd.DataFrame({
    "variable": categorical_cols,
    "levels": [cleaned[c].nunique(dropna=False) for c in categorical_cols],
    "most common level": [frequency_tables[c].index[0] for c in categorical_cols],
    "count": [int(frequency_tables[c].iloc[0]) for c in categorical_cols],
}))
make_body = cleaned.groupby(["make", "body-style"], observed=True).size().reset_index(name="cars")
fig = px.treemap(
    make_body, path=["make", "body-style"], values="cars",
    color="cars", color_continuous_scale="Tealgrn",
    title="Vehicle frequency: make → body style"
)
display(fig)
price_median = float(cleaned["price"].median())
price_mean = float(cleaned["price"].mean())
price_min, price_max = int(cleaned["price"].min()), int(cleaned["price"].max())
notes["q2_summary"] = (
    f"The styled table reports continuous-variable count, mean, spread, and quartiles; the "
    f"categorical table reports each field's level count and modal frequency. Price has "
    f"**mean ${price_mean:,.0f}, median ${price_median:,.0f}, and range "
    f"${price_min:,.0f}–${price_max:,.0f}**. The treemap encodes the make/body-style frequency "
    "counts, with larger rectangles representing more observations."
)


,count,mean,std,min,25%,50%,75%,max
symboling,201.00,0.84,1.25,-2.00,0.00,1.00,2.00,3.00
normalized-losses,201.00,122.00,32.00,65.00,101.00,122.00,137.00,256.00
wheel-base,201.00,98.80,6.07,86.60,94.50,97.00,102.40,120.90
length,201.00,0.84,0.06,0.68,0.80,0.83,0.88,1.00
width,201.00,0.92,0.03,0.84,0.89,0.91,0.93,1.00
height,201.00,53.77,2.45,47.80,52.00,54.10,55.50,59.80
curb-weight,201.00,"2,555.67",517.30,"1,488.00","2,169.00","2,414.00","2,926.00","4,066.00"
engine-size,201.00,126.88,41.55,61.00,98.00,120.00,141.00,326.00
bore,201.00,3.33,0.27,2.54,3.15,3.31,3.58,3.94
stroke,201.00,3.26,0.32,2.07,3.11,3.29,3.41,4.17


,variable,levels,most common level,count
0,make,22,toyota,32
1,aspiration,2,std,165
2,num-of-doors,2,four,115
3,body-style,5,sedan,94
4,drive-wheels,3,fwd,118
5,engine-location,2,front,198
6,engine-type,6,ohc,145
7,num-of-cylinders,7,four,157
8,fuel-system,8,mpfi,92
9,horsepower-binned,3,Low,116


The gradient-styled table summarizes continuous variables, while the frequency table lists categorical level counts and modes. In the treemap, area represents observed car counts within each make → body-style combination.

The styled table reports continuous-variable count, mean, spread, and quartiles; the categorical table reports each field's level count and modal frequency. Price has **mean $13,207, median $10,295, and range $5,118–$45,400**. The treemap encodes the make/body-style frequency counts, with larger rectangles representing more observations.

In [6]:

distribution_cols = ["price", "horsepower", "engine-size", "curb-weight", "city-mpg", "highway-mpg"]
for column in distribution_cols:
    fig = px.histogram(
        cleaned, x=column, marginal="box", nbins=25,
        color_discrete_sequence=[COLORS[0]],
        title=f"Distribution of {column} with marginal box plot",
        labels={column: column.replace("-", " ").title()}
    )
    display(fig)

correlation = cleaned[numeric_cols].corr(numeric_only=True)
fig = px.imshow(
    correlation, text_auto=".2f", aspect="auto",
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
    title="Pearson correlation among numeric variables"
)
display(fig)
notes["q2_distributions"] = (
    f"The six histograms show price and five vehicle measurements with marginal box plots; "
    f"the displayed price range is **${price_min:,.0f}–${price_max:,.0f}**. The heatmap uses "
    f"Pearson correlations; price's strongest positive numeric association is "
    f"`{correlation['price'].drop('price').idxmax()}` "
    f"(r = {correlation['price'].drop('price').max():.2f}), and its strongest negative is "
    f"`{correlation['price'].drop('price').idxmin()}` "
    f"(r = {correlation['price'].drop('price').min():.2f}). Correlation describes association, "
    "not causation."
)


The histograms and marginal box plots show skew, spread, and potential univariate extremes. The heatmap makes the direction and strength of numeric pairwise associations visible; the text identifies the actual strongest positive and negative associations with price.

The six histograms show price and five vehicle measurements with marginal box plots; the displayed price range is **$5,118–$45,400**. The heatmap uses Pearson correlations; price's strongest positive numeric association is `engine-size` (r = 0.87), and its strongest negative is `highway-mpg` (r = -0.70). Correlation describes association, not causation.

In [7]:

price_tier, tier_edges = pd.qcut(
    cleaned["price"], q=3, labels=["Economy", "Mid-range", "Luxury"],
    retbins=True, duplicates="drop"
)
cleaned["price_tier"] = price_tier
tier_profile = cleaned.groupby("price_tier", observed=True).agg(
    cars=("price", "size"),
    price_min=("price", "min"),
    price_median=("price", "median"),
    price_max=("price", "max"),
    mean_horsepower=("horsepower", "mean"),
    mean_engine_size=("engine-size", "mean"),
    mean_city_mpg=("city-mpg", "mean"),
    most_common_body_style=("body-style", lambda x: x.mode().iloc[0]),
    most_common_drive=("drive-wheels", lambda x: x.mode().iloc[0]),
)
display(tier_profile.round(2))
fig = px.box(
    cleaned, x="price_tier", y="price", color="price_tier",
    category_orders={"price_tier": ["Economy", "Mid-range", "Luxury"]},
    color_discrete_sequence=COLORS,
    title="Price tiers created from one-third quantiles"
)
display(fig)
notes["q2_tiers"] = (
    f"Quantile cut points are **${tier_edges[1]:,.0f}** and **${tier_edges[2]:,.0f}** "
    "(the observed minimum/maximum bound the outer tiers). The profile table shows each tier's "
    "size, price spread, average power/engine size/fuel economy, and modal body style/drive. "
    f"The median prices are Economy **${tier_profile.loc['Economy', 'price_median']:,.0f}**, "
    f"Mid-range **${tier_profile.loc['Mid-range', 'price_median']:,.0f}**, and "
    f"Luxury **${tier_profile.loc['Luxury', 'price_median']:,.0f}**. Profiles should be read "
    "as descriptive contrasts, not causal effects."
)


,cars,price_min,price_median,price_max,mean_horsepower,mean_engine_size,mean_city_mpg,most_common_body_style,most_common_drive
price_tier,,,,,,,,,
Economy,67,5118,7099.0,8249,70.94,96.76,31.45,hatchback,fwd
Mid-range,67,8358,10295.0,13845,99.66,119.58,24.43,sedan,fwd
Luxury,67,13860,18280.0,45400,139.61,164.28,19.66,sedan,rwd


The box plot compares the three price bands; the adjacent table characterizes each tier with its actual price range, median, power, engine size, fuel economy, and most common body/drive type.

Quantile cut points are **$8,322** and **$13,850** (the observed minimum/maximum bound the outer tiers). The profile table shows each tier's size, price spread, average power/engine size/fuel economy, and modal body style/drive. The median prices are Economy **$7,099**, Mid-range **$10,295**, and Luxury **$18,280**. Profiles should be read as descriptive contrasts, not causal effects.

In [8]:

def bootstrap_mean_ci(values: pd.Series, n_resamples: int = 1000, seed: int = RANDOM_STATE) -> tuple:
    """Percentile 95% bootstrap confidence interval for the arithmetic mean."""
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(n_resamples, len(values)), replace=True)
    means = samples.mean(axis=1)
    return float(np.mean(values)), float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))


def grouped_price_summary(group_column: str) -> pd.DataFrame:
    rows = []
    for group, part in cleaned.groupby(group_column, observed=True):
        mean, low, high = bootstrap_mean_ci(part["price"])
        rows.append({
            group_column: group, "n": len(part), "mean_price": mean,
            "ci_low": low, "ci_high": high, "median_price": part["price"].median()
        })
    return pd.DataFrame(rows).sort_values("mean_price", ascending=False)


by_body = grouped_price_summary("body-style")
by_drive = grouped_price_summary("drive-wheels")
display(by_body.round(1))
display(by_drive.round(1))

fig = go.Figure()
for i, row in by_body.reset_index(drop=True).iterrows():
    fig.add_trace(go.Bar(
        x=[row["body-style"]], y=[row["mean_price"]],
        name=row["body-style"], marker_color=COLORS[i % len(COLORS)],
        error_y=dict(
            type="data", symmetric=False,
            array=[row["ci_high"] - row["mean_price"]],
            arrayminus=[row["mean_price"] - row["ci_low"]]
        ),
        customdata=[[row["n"], row["ci_low"], row["ci_high"]]],
        hovertemplate="%{x}<br>Mean $%{y:,.0f}<br>n=%{customdata[0]}<br>"
                      "95% CI $%{customdata[1]:,.0f}–$%{customdata[2]:,.0f}<extra></extra>"
    ))
fig.update_layout(
    title="Mean price by body style with bootstrap 95% confidence intervals (1,000 resamples)",
    xaxis_title="Body style", yaxis_title="Mean price ($)", showlegend=False
)
display(fig)

body_drive_mean = cleaned.pivot_table(
    index="drive-wheels", columns="body-style", values="price", aggfunc="mean", observed=True
)
fig = px.imshow(
    body_drive_mean, text_auto=",.0f", aspect="auto",
    color_continuous_scale="YlOrRd", title="Mean price by drive wheels × body style",
    labels=dict(x="Body style", y="Drive wheels", color="Mean price ($)")
)
display(fig)
highest_body, lowest_body = by_body.iloc[0], by_body.iloc[-1]
highest_drive, lowest_drive = by_drive.iloc[0], by_drive.iloc[-1]
notes["q3_groups"] = (
    f"Means and percentile 95% bootstrap CIs use **1,000 resamples per group**. By body style, "
    f"the highest mean is **{highest_body['body-style']} (${highest_body.mean_price:,.0f}; "
    f"95% CI ${highest_body.ci_low:,.0f}–${highest_body.ci_high:,.0f}; n={int(highest_body.n)})** "
    f"and the lowest is **{lowest_body['body-style']} (${lowest_body.mean_price:,.0f}; "
    f"95% CI ${lowest_body.ci_low:,.0f}–${lowest_body.ci_high:,.0f}; n={int(lowest_body.n)})**. "
    f"By drive wheels, the highest mean is **{highest_drive['drive-wheels']} "
    f"(${highest_drive.mean_price:,.0f})**, and the lowest is **{lowest_drive['drive-wheels']} "
    f"(${lowest_drive.mean_price:,.0f})**. The heatmap shows the interaction cells; sparse cells "
    "should be interpreted cautiously."
)


,body-style,n,mean_price,ci_low,ci_high,median_price
1,hardtop,8,22208.5,13829.7,31308.5,19687.5
0,convertible,6,21890.5,14055.1,29917.7,17084.5
3,sedan,94,14459.8,12755.5,16135.2,11078.5
4,wagon,25,12372.0,10542.8,14575.6,11694.0
2,hatchback,68,9957.4,8949.3,10926.3,8672.0


,drive-wheels,n,mean_price,ci_low,ci_high,median_price
2,rwd,75,19757.6,17827.5,21787.3,16900.0
0,4wd,8,10241.0,8492.8,12763.0,9005.5
1,fwd,118,9244.8,8625.2,9862.7,8192.0


The two summary tables show group counts, means, medians, and bootstrap confidence intervals. The bar chart's error bars are the 95% bootstrap intervals; the heatmap displays mean prices for each drive-wheels × body-style cell.

Means and percentile 95% bootstrap CIs use **1,000 resamples per group**. By body style, the highest mean is **hardtop ($22,208; 95% CI $13,830–$31,308; n=8)** and the lowest is **hatchback ($9,957; 95% CI $8,949–$10,926; n=68)**. By drive wheels, the highest mean is **rwd ($19,758)**, and the lowest is **fwd ($9,245)**. The heatmap shows the interaction cells; sparse cells should be interpreted cautiously.

In [9]:

def kruskal_and_dunn(group_column: str):
    groups = [
        part["price"].to_numpy()
        for _, part in cleaned.groupby(group_column, observed=True)
        if len(part) >= 2
    ]
    labels = [
        label for label, part in cleaned.groupby(group_column, observed=True) if len(part) >= 2
    ]
    result = stats.kruskal(*groups)
    dunn = sp.posthoc_dunn(cleaned, val_col="price", group_col=group_column, p_adjust="bonferroni")
    return labels, result, dunn


body_labels, body_kw, body_dunn = kruskal_and_dunn("body-style")
drive_labels, drive_kw, drive_dunn = kruskal_and_dunn("drive-wheels")
display(pd.DataFrame([
    {"grouping": "body-style", "groups": len(body_labels), "H": body_kw.statistic,
     "p-value": body_kw.pvalue},
    {"grouping": "drive-wheels", "groups": len(drive_labels), "H": drive_kw.statistic,
     "p-value": drive_kw.pvalue},
]).round(5))
display(body_dunn.style.background_gradient(cmap="YlOrRd", axis=None).format("{:.4f}"))
display(drive_dunn.style.background_gradient(cmap="YlOrRd", axis=None).format("{:.4f}"))
body_sig = [
    f"{a} vs {b} (adjusted p={body_dunn.loc[a, b]:.4g})"
    for i, a in enumerate(body_dunn.index)
    for b in body_dunn.columns[i + 1:]
    if body_dunn.loc[a, b] < 0.05
]
drive_sig = [
    f"{a} vs {b} (adjusted p={drive_dunn.loc[a, b]:.4g})"
    for i, a in enumerate(drive_dunn.index)
    for b in drive_dunn.columns[i + 1:]
    if drive_dunn.loc[a, b] < 0.05
]
notes["q3_tests"] = (
    f"Kruskal–Wallis detects group-distribution differences for body style "
    f"(H={body_kw.statistic:.2f}, p={body_kw.pvalue:.3g}) and drive wheels "
    f"(H={drive_kw.statistic:.2f}, p={drive_kw.pvalue:.3g}). Dunn pairwise p-values are "
    "Bonferroni-adjusted; pairs below 0.05 are "
    + ("; ".join(body_sig) if body_sig else "none for body style")
    + " (body style), and "
    + ("; ".join(drive_sig) if drive_sig else "none for drive wheels")
    + " (drive wheels). Kruskal–Wallis tests distributions, not means; bootstrap intervals "
    "describe uncertainty in the group means."
)


,grouping,groups,H,p-value
0,body-style,5,25.02312,0.00005
1,drive-wheels,3,93.18890,0.00000


,convertible,hardtop,hatchback,sedan,wagon
convertible,1.0000,1.0000,0.0086,0.5447,0.4833
hardtop,1.0000,1.0000,0.0401,1.0000,1.0000
hatchback,0.0086,0.0401,1.0000,0.0013,0.2566
sedan,0.5447,1.0000,0.0013,1.0000,1.0000
wagon,0.4833,1.0000,0.2566,1.0000,1.0000


,4wd,fwd,rwd
4wd,1.0000,1.0000,0.0097
fwd,1.0000,1.0000,0.0000
rwd,0.0097,0.0000,1.0000


Kruskal–Wallis tests whether prices differ across each grouping without assuming normality. The two Dunn matrices show Bonferroni-adjusted pairwise p-values; the narrative identifies which comparisons pass the 0.05 threshold.

Kruskal–Wallis detects group-distribution differences for body style (H=25.02, p=4.98e-05) and drive wheels (H=93.19, p=5.81e-21). Dunn pairwise p-values are Bonferroni-adjusted; pairs below 0.05 are convertible vs hatchback (adjusted p=0.008591); hardtop vs hatchback (adjusted p=0.04005); hatchback vs sedan (adjusted p=0.001282) (body style), and 4wd vs rwd (adjusted p=0.009743); fwd vs rwd (adjusted p=1.735e-21) (drive wheels). Kruskal–Wallis tests distributions, not means; bootstrap intervals describe uncertainty in the group means.

In [10]:

numeric_features = [column for column in numeric_cols if column != "price"]
corr_rows = []
for column in numeric_features:
    corr_rows.append({
        "feature": column,
        "Pearson": cleaned[column].corr(cleaned["price"], method="pearson"),
        "Spearman": cleaned[column].corr(cleaned["price"], method="spearman"),
        "Kendall": cleaned[column].corr(cleaned["price"], method="kendall"),
    })
correlation_tests = pd.DataFrame(corr_rows).set_index("feature")
correlation_tests["|Pearson|"] = correlation_tests["Pearson"].abs()
correlation_tests = correlation_tests.sort_values("|Pearson|", ascending=False)
display(correlation_tests.drop(columns="|Pearson|").round(3))

lowess_features = ["engine-size", "horsepower", "curb-weight", "city-mpg"]
fig = make_subplots(
    rows=2, cols=2, subplot_titles=[f"{c} vs price" for c in lowess_features],
    horizontal_spacing=0.12, vertical_spacing=0.16
)
for idx, column in enumerate(lowess_features):
    row, col = divmod(idx, 2)
    row, col = row + 1, col + 1
    fig.add_trace(go.Scatter(
        x=cleaned[column], y=cleaned["price"], mode="markers",
        marker=dict(color=COLORS[0], opacity=0.65, size=6),
        name="Cars", showlegend=(idx == 0),
        hovertemplate=f"{column}=%{{x}}<br>price=$%{{y:,.0f}}<extra></extra>"
    ), row=row, col=col)
    smooth = sm.nonparametric.lowess(cleaned["price"], cleaned[column], frac=0.45, return_sorted=True)
    fig.add_trace(go.Scatter(
        x=smooth[:, 0], y=smooth[:, 1], mode="lines",
        line=dict(color=COLORS[2], width=3), name="LOWESS", showlegend=(idx == 0)
    ), row=row, col=col)
fig.update_layout(
    title="Continuous attributes vs price with LOWESS trendlines",
    height=720, template="plotly_white"
)
fig.update_xaxes(title_text="Attribute")
fig.update_yaxes(title_text="Price ($)")
display(fig)
strong_pos = correlation_tests["Pearson"].idxmax()
strong_neg = correlation_tests["Pearson"].idxmin()
notes["q4a"] = (
    f"The side-by-side table reports Pearson (linear), Spearman (monotonic rank), and Kendall "
    f"(rank concordance) associations with price. The largest positive Pearson correlation is "
    f"**{strong_pos} (r={correlation_tests.loc[strong_pos, 'Pearson']:.3f})**; the most negative "
    f"is **{strong_neg} (r={correlation_tests.loc[strong_neg, 'Pearson']:.3f})**. The four "
    "scatter panels pair observed cars with LOWESS smoothers, which reveal nonlinear patterns "
    "without imposing a straight-line form."
)


,Pearson,Spearman,Kendall
feature,,,
engine-size,0.872,0.828,0.661
curb-weight,0.834,0.914,0.744
horsepower,0.810,0.849,0.665
city-L/100km,0.790,0.831,0.661
width,0.751,0.812,0.639
highway-mpg,-0.705,-0.827,-0.652
length,0.691,0.810,0.636
city-mpg,-0.687,-0.831,-0.661
wheel-base,0.585,0.682,0.540


The table compares three correlation measures for every numeric predictor. Scatter plots show actual cars, while LOWESS curves highlight local trends; these associations are descriptive.

The side-by-side table reports Pearson (linear), Spearman (monotonic rank), and Kendall (rank concordance) associations with price. The largest positive Pearson correlation is **engine-size (r=0.872)**; the most negative is **highway-mpg (r=-0.705)**. The four scatter panels pair observed cars with LOWESS smoothers, which reveal nonlinear patterns without imposing a straight-line form.

In [11]:

cat_features = categorical_cols

def correlation_ratio(categories: pd.Series, measurements: pd.Series) -> float:
    valid = pd.DataFrame({"category": categories, "value": measurements}).dropna()
    grand_mean = valid["value"].mean()
    total = ((valid["value"] - grand_mean) ** 2).sum()
    if total == 0:
        return 0.0
    between = sum(
        len(group) * (group["value"].mean() - grand_mean) ** 2
        for _, group in valid.groupby("category", observed=True)
    )
    return float(np.sqrt(between / total))


eta_table = pd.DataFrame({
    "categorical feature": cat_features,
    "levels": [cleaned[c].nunique() for c in cat_features],
    "eta": [correlation_ratio(cleaned[c], cleaned["price"]) for c in cat_features],
}).sort_values("eta", ascending=False)
display(eta_table.round(3))

subplot_rows = (len(cat_features) + 1) // 2
fig = make_subplots(
    rows=subplot_rows, cols=2, subplot_titles=cat_features,
    vertical_spacing=0.07, horizontal_spacing=0.10
)
for idx, column in enumerate(cat_features):
    row, col = divmod(idx, 2)
    groups = cleaned.groupby(column, observed=True)["price"]
    for level, values in groups:
        fig.add_trace(go.Box(
            x=[str(level)] * len(values), y=values,
            name=str(level), legendgroup=column, showlegend=False,
            marker_color=COLORS[idx % len(COLORS)], boxpoints="outliers",
            hovertemplate=f"{column}={level}<br>price=$%{{y:,.0f}}<extra></extra>"
        ), row=row + 1, col=col + 1)
    fig.update_xaxes(title_text=column, row=row + 1, col=col + 1)
    fig.update_yaxes(title_text="Price ($)", row=row + 1, col=col + 1)
fig.update_layout(
    title="Price distributions across categorical levels",
    height=1700, template="plotly_white"
)
display(fig)
top_eta = eta_table.iloc[0]
low_eta = eta_table.iloc[-1]
notes["q4b"] = (
    f"Eta (correlation ratio) ranges from 0 to 1 and measures between-category price variation "
    f"without assuming ordered categories. The largest eta is **{top_eta['categorical feature']} "
    f"(η={top_eta.eta:.3f}; {int(top_eta.levels)} levels)**; the smallest is "
    f"**{low_eta['categorical feature']} (η={low_eta.eta:.3f})**. The faceted Plotly box plots "
    "show the group distributions and within-group spread; high eta is descriptive and may partly "
    "reflect many category levels."
)


,categorical feature,levels,eta
0,make,22,0.892
7,num-of-cylinders,7,0.793
9,horsepower-binned,3,0.688
4,drive-wheels,3,0.638
8,fuel-system,8,0.594
6,engine-type,6,0.449
3,body-style,5,0.396
5,engine-location,2,0.331
1,aspiration,2,0.180
10,diesel,2,0.110


Eta summarizes the degree to which price varies between levels of each categorical feature. The faceted box plots reveal medians, spread, and outlying observations for those same groups.

Eta (correlation ratio) ranges from 0 to 1 and measures between-category price variation without assuming ordered categories. The largest eta is **make (η=0.892; 22 levels)**; the smallest is **num-of-doors (η=0.042)**. The faceted Plotly box plots show the group distributions and within-group spread; high eta is descriptive and may partly reflect many category levels.

In [12]:

def cramers_v(left: pd.Series, right: pd.Series) -> float:
    table = pd.crosstab(left, right)
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    chi2_stat = stats.chi2_contingency(table, correction=False)[0]
    n = table.to_numpy().sum()
    return float(np.sqrt((chi2_stat / n) / min(table.shape[0] - 1, table.shape[1] - 1)))


cramers = pd.DataFrame(index=cat_features, columns=cat_features, dtype=float)
for left in cat_features:
    for right in cat_features:
        cramers.loc[left, right] = 1.0 if left == right else cramers_v(cleaned[left], cleaned[right])
display(cramers.round(3))
fig = px.imshow(
    cramers, text_auto=".2f", aspect="auto", zmin=0, zmax=1,
    color_continuous_scale="Tealrose", title="Cramér's V association across categorical pairs"
)
display(fig)

drive_body_counts = pd.crosstab(cleaned["drive-wheels"], cleaned["body-style"])
chi2_stat, chi2_p, chi2_dof, chi2_expected = stats.chi2_contingency(drive_body_counts)
drive_body_percent = pd.crosstab(
    cleaned["drive-wheels"], cleaned["body-style"], normalize="index"
) * 100
fig = px.bar(
    drive_body_percent.reset_index().melt(id_vars="drive-wheels", var_name="body-style", value_name="percent"),
    x="drive-wheels", y="percent", color="body-style", barmode="stack",
    color_discrete_sequence=COLORS, title="Body-style composition within drive-wheel groups",
    labels={"percent": "Share within drive-wheel group (%)"}
)
display(fig)
top_pair_values = cramers.where(np.triu(np.ones(cramers.shape), k=1).astype(bool)).stack()
top_pair = top_pair_values.idxmax()
notes["q4c"] = (
    f"Cramér's V ranges from 0 (no association) to 1 (strong association). The largest off-diagonal "
    f"pair is **{top_pair[0]} × {top_pair[1]} (V={top_pair_values.max():.3f})**. The stacked bar "
    f"shows body-style percentages within each drive-wheel category. Pearson's chi-square test for "
    f"drive-wheels × body-style gives **χ²({chi2_dof})={chi2_stat:.2f}, p={chi2_p:.3g}** "
    f"(N={int(drive_body_counts.to_numpy().sum())}); minimum expected count is "
    f"{chi2_expected.min():.2f}. Interpret the chi-square p-value cautiously if small expected "
    "counts violate the approximation."
)


,make,aspiration,num-of-doors,body-style,drive-wheels,engine-location,engine-type,num-of-cylinders,fuel-system,horsepower-binned,diesel,gas
make,1.000,0.521,0.439,0.461,0.684,0.864,0.741,0.631,0.623,0.594,0.487,0.487
aspiration,0.521,1.000,0.063,0.090,0.135,0.057,0.228,0.231,0.644,0.337,0.408,0.408
num-of-doors,0.439,0.063,1.000,0.765,0.139,0.142,0.255,0.228,0.313,0.099,0.187,0.187
body-style,0.461,0.090,0.765,1.000,0.269,0.459,0.210,0.205,0.241,0.189,0.224,0.224
drive-wheels,0.684,0.135,0.139,0.269,1.000,0.160,0.465,0.357,0.427,0.368,0.131,0.131
engine-location,0.864,0.057,0.142,0.459,0.160,1.000,0.433,0.334,0.134,0.342,0.041,0.041
engine-type,0.741,0.228,0.255,0.210,0.465,0.433,1.000,0.595,0.454,0.419,0.302,0.302
num-of-cylinders,0.631,0.231,0.228,0.205,0.357,0.334,0.595,1.000,0.411,0.536,0.245,0.245
fuel-system,0.623,0.644,0.313,0.241,0.427,0.134,0.454,0.411,1.000,0.567,1.000,1.000
horsepower-binned,0.594,0.337,0.099,0.189,0.368,0.342,0.419,0.536,0.567,1.000,0.140,0.140


The matrix displays Cramér's V for every categorical pair. The stacked percentages compare body-style composition across drive systems; the chi-square test assesses independence, with the smallest expected count provided as an assumption check.

Cramér's V ranges from 0 (no association) to 1 (strong association). The largest off-diagonal pair is **fuel-system × diesel (V=1.000)**. The stacked bar shows body-style percentages within each drive-wheel category. Pearson's chi-square test for drive-wheels × body-style gives **χ²(8)=29.11, p=0.000303** (N=201); minimum expected count is 0.24. Interpret the chi-square p-value cautiously if small expected counts violate the approximation.

In [13]:

target = cleaned["price"].astype(float)
drop_from_model = {"price", "city-L/100km", "horsepower-binned", "diesel", "gas", "price_tier"}
feature_cols = [c for c in cleaned.columns if c not in drop_from_model]
X = cleaned[feature_cols].copy()
model_numeric = X.select_dtypes(include=np.number).columns.tolist()
model_categorical = X.select_dtypes(exclude=np.number).columns.tolist()

def make_preprocessor(columns: list[str]) -> ColumnTransformer:
    num = [c for c in columns if c in model_numeric]
    cat = [c for c in columns if c in model_categorical]
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), num),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
            ]), cat),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )


preprocessor = make_preprocessor(feature_cols)
X_encoded = preprocessor.fit_transform(X)
encoded_names = preprocessor.get_feature_names_out()
lasso_scaler = StandardScaler(with_mean=False)
X_lasso = lasso_scaler.fit_transform(X_encoded)
y_scaled = StandardScaler().fit_transform(target.to_numpy().reshape(-1, 1)).ravel()
lasso = LassoCV(
    cv=5, alphas=np.logspace(-3, 1, 80),
    max_iter=50000, random_state=RANDOM_STATE
).fit(X_lasso, y_scaled)

def source_feature_indices(raw_features: list[str], names: np.ndarray) -> dict:
    mapping = {}
    for feature in raw_features:
        if feature in model_numeric:
            prefix = f"num__{feature}"
            idx = np.flatnonzero(names == prefix)
        else:
            prefix = f"cat__{feature}_"
            idx = np.flatnonzero(np.char.startswith(names.astype(str), prefix))
        mapping[feature] = idx
    return mapping


encoded_to_source = source_feature_indices(feature_cols, encoded_names)
lasso_source_score = {
    feature: float(np.max(np.abs(lasso.coef_[indices]))) if len(indices) else 0.0
    for feature, indices in encoded_to_source.items()
}
lasso_ranks = pd.Series(lasso_source_score).rank(ascending=False, method="min").astype(int)

path_alphas = np.logspace(1, -3, 80)
path_alphas, path_coefs, _ = lasso_path(X_lasso, y_scaled, alphas=path_alphas, max_iter=50000)
top_lasso_indices = np.argsort(np.abs(lasso.coef_))[::-1][:12]
fig = go.Figure()
for idx in top_lasso_indices:
    fig.add_trace(go.Scatter(
        x=path_alphas, y=path_coefs[idx], mode="lines",
        name=str(encoded_names[idx]), hovertemplate="%{fullData.name}<br>α=%{x:.4g}<br>coef=%{y:.3f}<extra></extra>"
    ))
fig.update_xaxes(type="log", title="Regularization strength α")
fig.update_yaxes(title="Standardized coefficient")
fig.update_layout(title="Lasso coefficient paths for the 12 strongest encoded predictors")
display(fig)
display(pd.DataFrame({
    "selected α (5-fold LassoCV)": [lasso.alpha_],
    "nonzero encoded coefficients": [int(np.count_nonzero(lasso.coef_))],
    "total encoded features": [len(encoded_names)],
}))
display(pd.Series(lasso_source_score).sort_values(ascending=False).head(12).rename("max |coefficient|").to_frame())
notes["q5_lasso"] = (
    f"The modeling predictors exclude price and the derived `city-L/100km`, "
    f"`horsepower-binned`, `diesel`, and `gas` columns to avoid target leakage or redundant encodings. "
    f"The ColumnTransformer creates **{len(model_numeric)} scaled numeric and "
    f"{len(model_categorical)} one-hot categorical source features**, producing "
    f"**{len(encoded_names)} encoded columns**. LassoCV selects α={lasso.alpha_:.4g} "
    f"and retains {int(np.count_nonzero(lasso.coef_))} nonzero encoded coefficients. "
    "The plot traces coefficient shrinkage as α increases; the table aggregates encoded levels "
    "back to source features."
)


,selected α (5-fold LassoCV),nonzero encoded coefficients,total encoded features
0,0.052665,15,72


,max |coefficient|
curb-weight,0.191626
engine-size,0.191075
width,0.166780
make,0.159329
horsepower,0.141594
engine-location,0.112534
num-of-cylinders,0.081181
drive-wheels,0.028330
body-style,0.022840
symboling,0.000000


The model pipeline imputes and scales numeric inputs and one-hot encodes categorical inputs. The Lasso path shows how encoded coefficients shrink under stronger regularization; the source-level table aggregates those encoded coefficients for interpretation.

The modeling predictors exclude price and the derived `city-L/100km`, `horsepower-binned`, `diesel`, and `gas` columns to avoid target leakage or redundant encodings. The ColumnTransformer creates **15 scaled numeric and 9 one-hot categorical source features**, producing **72 encoded columns**. LassoCV selects α=0.05266 and retains 15 nonzero encoded coefficients. The plot traces coefficient shrinkage as α increases; the table aggregates encoded levels back to source features.

In [14]:

rfecv_estimator = GradientBoostingRegressor(
    n_estimators=80, max_depth=2, learning_rate=0.05, random_state=RANDOM_STATE
)
rfecv = RFECV(
    estimator=rfecv_estimator, step=0.15, min_features_to_select=1,
    cv=KFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE),
    scoring="neg_mean_absolute_error", n_jobs=1
).fit(X_encoded, target)
rfecv_feature_ranks = {}
for feature, indices in encoded_to_source.items():
    rfecv_feature_ranks[feature] = (
        float(np.mean(rfecv.ranking_[indices])) if len(indices) else np.inf
    )
rfecv_ranks = pd.Series(rfecv_feature_ranks).rank(method="min").astype(int)
rfecv_cv = rfecv.cv_results_
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=rfecv_cv["n_features"], y=-rfecv_cv["mean_test_score"],
    mode="lines+markers", line=dict(color=COLORS[1], width=3),
    name="Mean CV MAE"
))
fig.add_vline(x=rfecv.n_features_, line_dash="dash", line_color=COLORS[2])
fig.update_layout(
    title="RFECV with Gradient Boosting: feature count vs cross-validated MAE",
    xaxis_title="Number of encoded features", yaxis_title="Cross-validated MAE ($)"
)
display(fig)
display(pd.DataFrame({
    "RFECV selected encoded features": [int(rfecv.n_features_)],
    "encoded feature count": [len(encoded_names)],
    "best mean CV MAE ($)": [-float(np.max(rfecv_cv["mean_test_score"]))],
}))
display(rfecv_ranks.sort_values().head(12).rename("aggregated RFECV rank").to_frame())
notes["q5_rfecv"] = (
    f"RFECV uses a Gradient Boosting Regressor and four-fold CV to minimize MAE. It selects "
    f"**{int(rfecv.n_features_)} of {len(encoded_names)} encoded columns**, with best mean "
    f"CV MAE **${-float(np.max(rfecv_cv['mean_test_score'])):,.0f}**. The line chart shows "
    "cross-validated error as features are recursively removed; source-feature ranks average "
    "the ranks of their one-hot/numeric columns."
)


,RFECV selected encoded features,encoded feature count,best mean CV MAE ($)
0,62,72,1619.681174


,aggregated RFECV rank
symboling,1
normalized-losses,1
aspiration,1
num-of-doors,1
drive-wheels,1
body-style,1
engine-location,1
wheel-base,1
curb-weight,1
length,1


The RFECV curve shows its internal cross-validated MAE at each feature-set size, and the dashed line marks the selected size. The source-level ranks combine encoded columns belonging to the same original field.

RFECV uses a Gradient Boosting Regressor and four-fold CV to minimize MAE. It selects **62 of 72 encoded columns**, with best mean CV MAE **$1,620**. The line chart shows cross-validated error as features are recursively removed; source-feature ranks average the ranks of their one-hot/numeric columns.

In [15]:

gbr_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=RANDOM_STATE),
    param_grid={
        "n_estimators": [80, 140],
        "max_depth": [2, 3],
        "learning_rate": [0.05, 0.1],
    },
    scoring="neg_mean_absolute_error",
    cv=KFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=1,
    refit=True,
).fit(X_encoded, target)
tuned_gbr = gbr_grid.best_estimator_
explainer = shap.TreeExplainer(tuned_gbr)
shap_values_encoded = np.asarray(explainer.shap_values(X_encoded))
shap_source_values = {}
shap_source_scores = {}
for feature, indices in encoded_to_source.items():
    if len(indices):
        shap_source_values[feature] = shap_values_encoded[:, indices].sum(axis=1)
        shap_source_scores[feature] = float(
            np.mean(np.abs(shap_values_encoded[:, indices]).sum(axis=1))
        )
shap_ranks = pd.Series(shap_source_scores).rank(ascending=False, method="min").astype(int)
top_shap_features = pd.Series(shap_source_scores).sort_values(ascending=False).head(10).index.tolist()

summary_fig = go.Figure()
for rank, feature in enumerate(top_shap_features):
    raw_values = X[feature]
    if pd.api.types.is_numeric_dtype(raw_values):
        marker_color = raw_values.to_numpy(dtype=float)
        colorbar = dict(title=feature) if rank == 0 else None
        color_scale = "Viridis"
    else:
        codes, _ = pd.factorize(raw_values.astype(str), sort=True)
        marker_color = codes
        colorbar = dict(title=f"{feature} level code") if rank == 0 else None
        color_scale = "Turbo"
    jitter = np.random.default_rng(RANDOM_STATE + rank).normal(0, 0.08, len(X))
    summary_fig.add_trace(go.Scatter(
        x=shap_source_values[feature],
        y=np.full(len(X), rank) + jitter,
        mode="markers",
        marker=dict(
            size=6, opacity=0.68, color=marker_color,
            colorscale=color_scale, showscale=(rank == 0), colorbar=colorbar
        ),
        name=feature,
        customdata=np.column_stack([raw_values.astype(str), target]),
        hovertemplate=f"{feature}=%{{customdata[0]}}<br>SHAP=%{{x:.2f}}<br>"
                      "price=$%{customdata[1]:,.0f}<extra></extra>"
    ))
summary_fig.update_yaxes(
    tickmode="array", tickvals=list(range(len(top_shap_features))),
    ticktext=top_shap_features, autorange="reversed", title="Source feature"
)
summary_fig.update_xaxes(title="Aggregated SHAP contribution to predicted price ($)")
summary_fig.update_layout(
    title="SHAP summary: source-level contributions (interactive)",
    showlegend=False, height=650
)
display(summary_fig)

dependence_features = top_shap_features[:4]
dependence_fig = make_subplots(
    rows=2, cols=2, subplot_titles=[f"{feature} dependence" for feature in dependence_features],
    horizontal_spacing=0.13, vertical_spacing=0.18
)
for idx, feature in enumerate(dependence_features):
    row, col = divmod(idx, 2)
    row, col = row + 1, col + 1
    value = X[feature]
    dependence_fig.add_trace(go.Scatter(
        x=value.astype(str) if not pd.api.types.is_numeric_dtype(value) else value,
        y=shap_source_values[feature],
        mode="markers",
        marker=dict(
            size=7, opacity=0.7, color=target, colorscale="Viridis",
            showscale=(idx == 0), colorbar=dict(title="Observed price ($)") if idx == 0 else None
        ),
        customdata=np.column_stack([value.astype(str), target]),
        hovertemplate=f"{feature}=%{{customdata[0]}}<br>SHAP=%{{y:.2f}}<br>"
                      "observed price=$%{customdata[1]:,.0f}<extra></extra>"
    ), row=row, col=col)
    dependence_fig.update_xaxes(title_text=feature, row=row, col=col)
    dependence_fig.update_yaxes(title_text="SHAP contribution ($)", row=row, col=col)
dependence_fig.update_layout(
    title="SHAP dependence plots for the four highest-ranked source features",
    height=720, showlegend=False
)
display(dependence_fig)
display(pd.DataFrame({
    "best parameters": [gbr_grid.best_params_],
    "best 4-fold CV MAE ($)": [-gbr_grid.best_score_],
    "top SHAP feature": [top_shap_features[0]],
    "top mean |SHAP| ($)": [shap_source_scores[top_shap_features[0]]],
}))
notes["q5_shap"] = (
    f"Grid search selects **{gbr_grid.best_params_}** (best four-fold MAE "
    f"${-gbr_grid.best_score_:,.0f}). The source-level SHAP summary ranks "
    f"**{top_shap_features[0]}** first (mean absolute contribution "
    f"${shap_source_scores[top_shap_features[0]]:,.0f}), followed by "
    f"{', '.join(top_shap_features[1:4])}. Points to the right increase the fitted prediction "
    "relative to the model's baseline; dependence panels relate each feature's observed value "
    "to its contribution. These are model explanations, not causal effects."
)


,best parameters,best 4-fold CV MAE ($),top SHAP feature,top mean |SHAP| ($)
0,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti...",1479.805457,engine-size,2492.783977


The tuned Gradient Boosting model is explained with source-level SHAP contributions. The interactive summary shows both positive and negative contributions; the dependence plots show how contributions vary with observed feature values.

Grid search selects **{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 140}** (best four-fold MAE $1,480). The source-level SHAP summary ranks **engine-size** first (mean absolute contribution $2,493), followed by curb-weight, horsepower, highway-mpg. Points to the right increase the fitted prediction relative to the model's baseline; dependence panels relate each feature's observed value to its contribution. These are model explanations, not causal effects.

In [16]:

all_feature_ranks = pd.DataFrame({
    "Lasso rank": lasso_ranks,
    "RFECV rank": rfecv_ranks,
    "SHAP rank": shap_ranks,
}).reindex(feature_cols)
all_feature_ranks["mean rank"] = all_feature_ranks.mean(axis=1)
all_feature_ranks["top-5 votes"] = (
    (all_feature_ranks[["Lasso rank", "RFECV rank", "SHAP rank"]] <= 5).sum(axis=1)
)
consensus = all_feature_ranks.sort_values(
    ["mean rank", "top-5 votes"], ascending=[True, False]
)
display(consensus.head(15).round(2))
reduced_features = consensus.head(8).index.tolist()
top5_sets = {
    "Lasso": set(lasso_ranks.nsmallest(5).index),
    "RFECV": set(rfecv_ranks.nsmallest(5).index),
    "SHAP": set(shap_ranks.nsmallest(5).index),
}
shared_top5 = sorted(set.intersection(*top5_sets.values()))
notes["q5_consensus"] = (
    f"The consensus ranks raw predictors by the mean of their three method-specific ranks; "
    f"**{len(shared_top5)} predictors** occur in all three top-five lists"
    + (f": {', '.join(shared_top5)}." if shared_top5 else ".")
    + f" The reduced model uses the eight best consensus-ranked source features: "
    f"{', '.join(reduced_features)}. Agreement is useful evidence of robustness, while "
    "disagreement reflects different assumptions: shrinkage, recursive predictive contribution, "
    "and tree-model attribution."
)


,Lasso rank,RFECV rank,SHAP rank,mean rank,top-5 votes
curb-weight,1,1,2,1.33,3
engine-size,2,1,1,1.33,3
horsepower,5,1,3,3.00,3
width,3,1,8,4.00,2
highway-mpg,10,1,4,5.00,2
wheel-base,10,1,5,5.33,2
length,10,1,7,6.00,1
body-style,9,1,9,6.33,1
bore,10,1,10,7.00,1
drive-wheels,8,1,13,7.33,1


The table places the three selection methods side by side, including their mean rank and number of top-five votes. Features selected by all three are explicitly identified; the reduced model uses the eight highest consensus-ranked original predictors.

The consensus ranks raw predictors by the mean of their three method-specific ranks; **0 predictors** occur in all three top-five lists. The reduced model uses the eight best consensus-ranked source features: curb-weight, engine-size, horsepower, width, highway-mpg, wheel-base, length, body-style. Agreement is useful evidence of robustness, while disagreement reflects different assumptions: shrinkage, recursive predictive contribution, and tree-model attribution.

In [17]:

# Repeated CV evaluates complete preprocessing/model pipelines; no preprocessing is fitted outside
# each fold. The consensus subset itself was chosen on the complete dataset (see limitation below).
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
model_params = {
    "n_estimators": [80, 140],
    "max_depth": [2, 3],
    "learning_rate": [0.05, 0.1],
}
full_pipeline = Pipeline([
    ("preprocess", make_preprocessor(feature_cols)),
    ("model", GradientBoostingRegressor(random_state=RANDOM_STATE, **gbr_grid.best_params_))
])
reduced_pipeline = Pipeline([
    ("preprocess", make_preprocessor(reduced_features)),
    ("model", GradientBoostingRegressor(random_state=RANDOM_STATE, **gbr_grid.best_params_))
])
scoring = {"R2": "r2", "MAE": "neg_mean_absolute_error"}
full_scores = cross_validate(
    full_pipeline, X[feature_cols], target, cv=cv, scoring=scoring,
    n_jobs=1, return_train_score=False
)
reduced_scores = cross_validate(
    reduced_pipeline, X[reduced_features], target, cv=cv, scoring=scoring,
    n_jobs=1, return_train_score=False
)
cv_table = pd.DataFrame([
    {
        "feature set": "Full",
        "features": len(feature_cols),
        "CV R² mean": full_scores["test_R2"].mean(),
        "CV R² SD": full_scores["test_R2"].std(ddof=1),
        "CV MAE mean ($)": -full_scores["test_MAE"].mean(),
        "CV MAE SD ($)": full_scores["test_MAE"].std(ddof=1),
    },
    {
        "feature set": "Consensus reduced",
        "features": len(reduced_features),
        "CV R² mean": reduced_scores["test_R2"].mean(),
        "CV R² SD": reduced_scores["test_R2"].std(ddof=1),
        "CV MAE mean ($)": -reduced_scores["test_MAE"].mean(),
        "CV MAE SD ($)": reduced_scores["test_MAE"].std(ddof=1),
    },
])
display(cv_table.round(3))
better_row = cv_table.sort_values("CV MAE mean ($)").iloc[0]
notes["q5_cv"] = (
    f"On **{cv.get_n_splits()} repeated five-fold validation splits**, the full model "
    f"({len(feature_cols)} source features) averages R²={cv_table.loc[0, 'CV R² mean']:.3f} "
    f"(SD {cv_table.loc[0, 'CV R² SD']:.3f}) and MAE "
    f"${cv_table.loc[0, 'CV MAE mean ($)']:,.0f} (SD ${cv_table.loc[0, 'CV MAE SD ($)']:,.0f}); "
    f"the eight-feature consensus model averages R²={cv_table.loc[1, 'CV R² mean']:.3f} "
    f"(SD {cv_table.loc[1, 'CV R² SD']:.3f}) and MAE "
    f"${cv_table.loc[1, 'CV MAE mean ($)']:,.0f} (SD ${cv_table.loc[1, 'CV MAE SD ($)']:,.0f}). "
    f"By mean MAE, **{better_row['feature set']}** performs better. Preprocessing is refit within "
    "each fold, but the consensus feature subset was identified using the complete dataset before "
    "this comparison; therefore these CV scores are exploratory and may be optimistic. A fully "
    "unbiased final estimate would repeat all three selection methods inside nested outer folds."
)


,feature set,features,CV R² mean,CV R² SD,CV MAE mean ($),CV MAE SD ($)
0,Full,24,0.91,0.048,1444.958,190.381
1,Consensus reduced,8,0.91,0.039,1461.033,225.307


This table compares out-of-fold R² and MAE for full and reduced feature sets, including fold-to-fold variability. The preprocessing is refit per fold; the limitation concerning when the reduced feature set was chosen is important when interpreting model performance.

On **15 repeated five-fold validation splits**, the full model (24 source features) averages R²=0.910 (SD 0.048) and MAE $1,445 (SD $190); the eight-feature consensus model averages R²=0.910 (SD 0.039) and MAE $1,461 (SD $225). By mean MAE, **Full** performs better. Preprocessing is refit within each fold, but the consensus feature subset was identified using the complete dataset before this comparison; therefore these CV scores are exploratory and may be optimistic. A fully unbiased final estimate would repeat all three selection methods inside nested outer folds.

In [18]:

best_model = Pipeline([
    ("preprocess", make_preprocessor(reduced_features)),
    ("model", GradientBoostingRegressor(random_state=RANDOM_STATE, **gbr_grid.best_params_))
]).fit(X[reduced_features], target)
importance_values = best_model.named_steps["model"].feature_importances_
transformed_reduced_names = best_model.named_steps["preprocess"].get_feature_names_out()
reduced_mapping = source_feature_indices(reduced_features, transformed_reduced_names)
source_importance = {
    feature: float(np.sum(importance_values[indices]))
    for feature, indices in reduced_mapping.items()
}
importance_table = pd.Series(source_importance).sort_values(ascending=False).rename(
    "normalized tree importance"
).to_frame()
display(importance_table.round(3))

baseline = float(np.median(target))
test_cars = X[reduced_features].copy()
test_cars["predicted_price"] = best_model.predict(X[reduced_features])
test_cars["observed_price"] = target.to_numpy()
test_cars["pricing_rule"] = test_cars["predicted_price"].round(-2)
display(test_cars[["predicted_price", "observed_price", "pricing_rule"]].head(8).round(0))
mae_rule = float(cv_table.loc[1, "CV MAE mean ($)"])
notes["q5_recommendation"] = (
    f"Dealer recommendation: use the **eight-feature consensus Gradient Boosting model** as a "
    f"starting-point estimate, then review comparable local listings and vehicle condition. Its "
    f"mean CV MAE is about **${mae_rule:,.0f}**, so treat a prediction as a negotiation anchor with "
    f"roughly that average absolute historical error—not as a guaranteed sale price. A simple rule "
    f"of thumb is to price a car near its model-predicted value (rounded to the nearest $100), "
    f"adjusting within about **±${mae_rule:,.0f}** for condition, mileage/history not captured here, "
    "and current local demand. Limitations: only 201 cars from a historical dataset; sparse model/"
    "category groups; no mileage, condition, age, location, or time effects; associations are not "
    "causal; feature selection used all records before CV, so performance may be optimistic; and "
    "the sample's market prices may not generalize to today's inventory."
)


,normalized tree importance
engine-size,0.639
curb-weight,0.150
horsepower,0.088
highway-mpg,0.062
wheel-base,0.019
width,0.018
length,0.017
body-style,0.007


,predicted_price,observed_price,pricing_rule
0,14979.0,13495.0,15000.0
1,14979.0,16500.0,15000.0
2,16090.0,16500.0,16100.0
3,13866.0,13950.0,13900.0
4,17178.0,17450.0,17200.0
5,14537.0,15250.0,14500.0
6,18468.0,17710.0,18500.0
7,18413.0,18920.0,18400.0


The model-based pricing rule rounds a predicted price to the nearest hundred dollars and treats the consensus model's cross-validated MAE as an approximate uncertainty band. Use it as a starting point alongside inspection and current market comparables, not as a guaranteed price.

Dealer recommendation: use the **eight-feature consensus Gradient Boosting model** as a starting-point estimate, then review comparable local listings and vehicle condition. Its mean CV MAE is about **$1,461**, so treat a prediction as a negotiation anchor with roughly that average absolute historical error—not as a guaranteed sale price. A simple rule of thumb is to price a car near its model-predicted value (rounded to the nearest $100), adjusting within about **±$1,461** for condition, mileage/history not captured here, and current local demand. Limitations: only 201 cars from a historical dataset; sparse model/category groups; no mileage, condition, age, location, or time effects; associations are not causal; feature selection used all records before CV, so performance may be optimistic; and the sample's market prices may not generalize to today's inventory.